In [ ]:
import json

with open('uniadilr/abduction.jsonl', 'r', encoding='utf-8') as f:
    length_list = []
    for line in f:
        if line.strip():
            record = json.loads(line)
            length_list.append(record["proof"].count('&')+1)
            if len(length_list)==1:
                print(line)

for i in range(1, 11):
    print(f"number of answers with {i} sentences: {length_list.count(i)}")
    

In [ ]:
import json
import random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from typing import List, Dict, Tuple, Set
from tqdm import tqdm

def parse_proof(proof: str) -> List[str]:
    """Extract sentence IDs from proof string."""
    # Extract the sentence part before '->'
    sent_part = proof.split('->')[0].strip()
    # Split by '&' and clean up
    sentences = [s.strip() for s in sent_part.split('&')]
    return sentences

def get_sentence_text(context: Dict[str, str], sent_ids: List[str]) -> str:
    """Get combined text for given sentence IDs."""
    return ' '.join([context[sid] for sid in sent_ids])

def find_similar_sentences(
    context: Dict[str, str],
    ground_truth_sents: List[str],
    top_n: int = 5
) -> List[str]:
    """Find top N similar sentences using TF-IDF, excluding ground truth."""
    # Get all sentence IDs
    all_sent_ids = list(context.keys())
    
    # Remove ground truth sentences
    candidate_sent_ids = [sid for sid in all_sent_ids if sid not in ground_truth_sents]
    
    if len(candidate_sent_ids) < top_n:
        # If we don't have enough candidates, return all available
        return candidate_sent_ids
    
    # Get text for ground truth and candidates
    gt_text = get_sentence_text(context, ground_truth_sents)
    candidate_texts = [context[sid] for sid in candidate_sent_ids]
    
    # Create TF-IDF vectors
    all_texts = [gt_text] + candidate_texts
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(all_texts)
    
    # Calculate cosine similarity
    similarities = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
    
    # Get top N indices
    top_indices = np.argsort(similarities)[-top_n:][::-1]
    
    # Return corresponding sentence IDs
    return [candidate_sent_ids[i] for i in top_indices]

def format_option(sent_ids: List[str]) -> str:
    """Format sentence IDs as option string."""
    # Sort to ensure consistent formatting
    return ' & '.join(sorted(sent_ids, key=lambda x: int(x.replace('sent', ''))))

def normalize_option(option: str) -> str:
    """Normalize option for comparison (sort sentence IDs)."""
    parts = [s.strip() for s in option.split('&')]
    return format_option(parts)

def generate_random_option(
    similar_sents: List[str],
    num_sents: int,
    used_options: Set[str],
    max_attempts: int = 50
) -> str:
    """Generate a random option that hasn't been used yet."""
    for _ in range(max_attempts):
        if num_sents <= len(similar_sents):
            selected = random.sample(similar_sents, num_sents)
        else:
            selected = similar_sents
        
        option = format_option(selected)
        if option not in used_options:
            return option
    
    # If we couldn't find a unique option, return the best we got
    # This is a fallback - in practice, we have enough diversity
    return format_option(random.sample(similar_sents, min(num_sents, len(similar_sents))))

def create_options_case1(
    ground_truth: List[str],
    similar_sents: List[str],
    context: Dict[str, str]
) -> List[str]:
    """Create options for case 1 (1 ground truth sentence)."""
    options = []
    used_options = set()
    
    # Option 1: Ground truth
    opt1 = format_option(ground_truth)
    options.append(opt1)
    used_options.add(opt1)
    
    # Get all available sentences (similar + others)
    all_available = list(context.keys())
    all_available = [s for s in all_available if s not in ground_truth]
    
    # Option 2: One random from similar
    opt2 = generate_random_option(similar_sents, 1, used_options)
    options.append(opt2)
    used_options.add(opt2)
    
    # Option 3: Two random from similar
    opt3 = generate_random_option(similar_sents, 2, used_options)
    options.append(opt3)
    used_options.add(opt3)
    
    # Option 4: 50% one, 50% two
    num_sents = 1 if random.random() < 0.5 else 2
    opt4 = generate_random_option(similar_sents if len(similar_sents) >= num_sents else all_available, 
                                   num_sents, used_options)
    
    # If still duplicate, try with all available sentences
    attempts = 0
    while opt4 in used_options and attempts < 20:
        num_sents = random.choice([1, 2, 3])
        opt4 = generate_random_option(all_available, num_sents, used_options)
        attempts += 1
    
    options.append(opt4)
    used_options.add(opt4)
    
    return options

def create_options_case2(
    ground_truth: List[str],
    similar_sents: List[str],
    context: Dict[str, str]
) -> List[str]:
    """Create options for case 2 (2 ground truth sentences)."""
    options = []
    used_options = set()
    
    # Option 1: Ground truth
    opt1 = format_option(ground_truth)
    options.append(opt1)
    used_options.add(opt1)
    
    # Get all available sentences
    all_available = list(context.keys())
    all_available = [s for s in all_available if s not in ground_truth]
    
    # Option 2: One random from ground truth
    opt2 = format_option([random.choice(ground_truth)])
    
    # Ensure uniqueness
    attempts = 0
    while opt2 in used_options and attempts < 20:
        subset_size = random.choice([1, 2]) if len(ground_truth) >= 2 else 1
        opt2 = format_option(random.sample(ground_truth, subset_size))
        attempts += 1
    
    options.append(opt2)
    used_options.add(opt2)
    
    # Option 3: Ground truth + one random from similar
    opt3_sents = ground_truth + [random.choice(similar_sents)]
    opt3 = format_option(opt3_sents)
    
    # Ensure uniqueness
    attempts = 0
    while opt3 in used_options and attempts < 20:
        if similar_sents:
            additional = random.sample(similar_sents, min(random.randint(1, 2), len(similar_sents)))
            opt3_sents = ground_truth + additional
        else:
            opt3_sents = ground_truth + [random.choice(all_available)]
        opt3 = format_option(opt3_sents)
        attempts += 1
    
    options.append(opt3)
    used_options.add(opt3)
    
    # Option 4: 50% one, 50% two from similar
    num_sents = 1 if random.random() < 0.5 else 2
    opt4 = generate_random_option(similar_sents if len(similar_sents) >= num_sents else all_available,
                                   num_sents, used_options)
    
    # If still duplicate, try with all available sentences
    attempts = 0
    while opt4 in used_options and attempts < 20:
        num_sents = random.choice([1, 2, 3])
        opt4 = generate_random_option(all_available, num_sents, used_options)
        attempts += 1
    
    options.append(opt4)
    used_options.add(opt4)
    
    return options

def create_options_case3(
    ground_truth: List[str],
    similar_sents: List[str],
    context: Dict[str, str]
) -> List[str]:
    """Create options for case 3 (3 ground truth sentences)."""
    options = []
    used_options = set()
    
    # Option 1: Ground truth
    opt1 = format_option(ground_truth)
    options.append(opt1)
    used_options.add(opt1)
    
    # Get all available sentences
    all_available = list(context.keys())
    all_available = [s for s in all_available if s not in ground_truth]
    
    # Option 2: 50% one, 50% two from ground truth
    subset_size = 1 if random.random() < 0.5 else 2
    opt2 = format_option(random.sample(ground_truth, subset_size))
    
    # Ensure uniqueness
    attempts = 0
    while opt2 in used_options and attempts < 20:
        subset_size = random.choice([1, 2])
        opt2 = format_option(random.sample(ground_truth, min(subset_size, len(ground_truth))))
        attempts += 1
    
    options.append(opt2)
    used_options.add(opt2)
    
    # Option 3: Ground truth + one random from similar
    opt3_sents = ground_truth + [random.choice(similar_sents)] if similar_sents else ground_truth
    opt3 = format_option(opt3_sents)
    
    # Ensure uniqueness
    attempts = 0
    while opt3 in used_options and attempts < 20:
        if similar_sents:
            additional = random.sample(similar_sents, min(random.randint(1, 2), len(similar_sents)))
            opt3_sents = ground_truth + additional
        else:
            additional = random.sample(all_available, min(random.randint(1, 2), len(all_available)))
            opt3_sents = ground_truth + additional
        opt3 = format_option(opt3_sents)
        attempts += 1
    
    options.append(opt3)
    used_options.add(opt3)
    
    # Option 4: 33% one, 33% two, 33% three from similar
    rand_val = random.random()
    if rand_val < 0.33:
        num_sents = 1
    elif rand_val < 0.66:
        num_sents = 2
    else:
        num_sents = 3
    
    opt4 = generate_random_option(similar_sents if len(similar_sents) >= num_sents else all_available,
                                   num_sents, used_options)
    
    # If still duplicate, try with all available sentences and different counts
    attempts = 0
    while opt4 in used_options and attempts < 20:
        num_sents = random.choice([1, 2, 3, 4])
        opt4 = generate_random_option(all_available, num_sents, used_options)
        attempts += 1
    
    options.append(opt4)
    used_options.add(opt4)
    
    return options

def format_context(context: Dict[str, str]) -> str:
    """Format context dictionary as a beautiful string."""
    formatted_lines = []
    for key in sorted(context.keys(), key=lambda x: int(x.replace('sent', ''))):
        formatted_lines.append(f"{key}: {context[key]}")
    return '\n'.join(formatted_lines)

def process_entry(entry: Dict) -> Dict:
    """Process a single JSONL entry and create the new format."""
    context = entry['context']
    hypothesis = entry['hypothesis']
    proof = entry['proof']
    
    # Parse ground truth sentences
    ground_truth = parse_proof(proof)
    num_gt = len(ground_truth)
    
    # Find similar sentences
    similar_sents = find_similar_sentences(context, ground_truth, top_n=10)
    
    # Create options based on number of ground truth sentences
    if num_gt == 1:
        options = create_options_case1(ground_truth, similar_sents, context)
    elif num_gt == 2:
        options = create_options_case2(ground_truth, similar_sents, context)
    elif num_gt == 3:
        options = create_options_case3(ground_truth, similar_sents, context)
    else:
        raise ValueError(f"Unexpected number of ground truth sentences: {num_gt}")
    
    # Verify we have 4 unique options
    if len(set(options)) < 4:
        # Add fallback options if needed
        all_sents = [s for s in context.keys() if s not in ground_truth]
        used = set(options)
        while len(set(options)) < 4 and all_sents:
            new_opt = format_option(random.sample(all_sents, min(random.randint(1, 3), len(all_sents))))
            if new_opt not in used:
                options.append(new_opt)
                used.add(new_opt)
    
    # Take only first 4 unique options
    unique_options = []
    seen = set()
    for opt in options:
        if opt not in seen:
            unique_options.append(opt)
            seen.add(opt)
        if len(unique_options) == 4:
            break
    
    # IMPORTANT: Remember the correct answer BEFORE shuffling
    correct_answer = format_option(ground_truth)
    
    # NOW shuffle the options randomly
    random.shuffle(unique_options)
    
    # Find which position (A, B, C, or D) the correct answer ended up in
    answer_labels = ['A', 'B', 'C', 'D']
    try:
        correct_index = unique_options.index(correct_answer)
        answer_label = answer_labels[correct_index]
    except ValueError:
        # Fallback if somehow correct answer is not in options
        answer_label = 'A'
    
    # Create the new entry with shuffled options
    new_entry = {
        'context': format_context(context),
        'question': f"Based on the provided context, which set of sentences justify or prove the validity of the hypothesis? hypothesis: {hypothesis}",
        'hypothesis': hypothesis,
        'options': {
            'A': unique_options[0] if len(unique_options) > 0 else '',
            'B': unique_options[1] if len(unique_options) > 1 else '',
            'C': unique_options[2] if len(unique_options) > 2 else '',
            'D': unique_options[3] if len(unique_options) > 3 else ''
        },
        'answer': answer_label
    }
    
    return new_entry

def count_lines(filename: str) -> int:
    """Count total lines in file for progress bar."""
    with open(filename, 'r', encoding='utf-8') as f:
        return sum(1 for _ in f)

def process_jsonl_file(input_file: str, output_file: str):
    """Process entire JSONL file with progress bar."""
    # Count total lines first
    print("Counting total entries...")
    total_lines = count_lines(input_file)
    print(f"Found {total_lines} entries to process.\n")
    
    # Track answer distribution
    answer_distribution = {'A': 0, 'B': 0, 'C': 0, 'D': 0}
    
    # Process with progress bar
    with open(input_file, 'r', encoding='utf-8') as f_in, \
         open(output_file, 'w', encoding='utf-8') as f_out:
        
        # Create progress bar
        pbar = tqdm(total=total_lines, desc="Processing entries", unit="entry")
        
        errors = 0
        for line_num, line in enumerate(f_in, 1):
            try:
                entry = json.loads(line.strip())
                new_entry = process_entry(entry)
                f_out.write(json.dumps(new_entry, ensure_ascii=False) + '\n')
                
                # Track answer distribution
                answer_distribution[new_entry['answer']] += 1
                
            except Exception as e:
                errors += 1
                tqdm.write(f"Error on line {line_num}: {str(e)}")
                
            finally:
                pbar.update(1)
        
        pbar.close()
    
    # Calculate percentages
    successful = total_lines - errors
    print(f"\n{'='*60}")
    print(f"Processing complete!")
    print(f"Total entries processed: {total_lines}")
    print(f"Errors encountered: {errors}")
    print(f"Success rate: {(successful / total_lines * 100):.2f}%")
    print(f"\nAnswer distribution (should be roughly 25% each):")
    for label in ['A', 'B', 'C', 'D']:
        count = answer_distribution[label]
        percentage = (count / successful * 100) if successful > 0 else 0
        print(f"  {label}: {count:5d} ({percentage:5.2f}%)")
    print(f"Output saved to: {output_file}")
    print(f"{'='*60}")

# Example usage
if __name__ == "__main__":
    input_file = "uniadilr/abduction.jsonl"
    output_file = "uniadilr/abduction_multi_choice.jsonl"
    
    print("="*60)
    print("JSONL Dataset Transformer")
    print("="*60)
    print(f"Input file: {input_file}")
    print(f"Output file: {output_file}")
    print("="*60 + "\n")
    
    process_jsonl_file(input_file, output_file)
